# 10-714 第四次作业扩展

本次作业是第四次作业的扩展，你将实现 Transformer 架构。在此作业中，所有需要实现的部分都在文件 `python/needle/nn/nn_transformer.py` 中。needle 库中的其他部分保持不变。本次扩展作业基于第四次作业，因此请确保从第四次作业中复制解决方案。

In [ ]:
# 设置作业的代码
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsyscourse/hw4_extra.git
%cd /content/drive/MyDrive/10714/hw4_extra

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

In [ ]:
# 必填项：MUGRADE API 密钥
MY_API_KEY = "<FILL YOUR API KEY HERE>"

In [ ]:
!make

In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [ ]:
import sys
sys.path.append('./python')

In [ ]:
# 下载 PTB 数据集
import urllib.request
import os

!mkdir -p './data/ptb'
# 下载 Penn Treebank 数据集
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

## Transformer

在前面的一次作业中，你已经实现了两个序列模型：循环神经网络（Recurrent Neural Network）和长短期记忆网络（Long Short-Term Memory）。这些模型曾是最先进且默认的序列建模（包括语言生成）架构选择，直到 2017 年著名的论文《Attention Is All You Need》（Vaswani 等，2017）问世。自那时起，Transformer——上述论文引入的模型架构——已成为语言任务上最标准、性能最好的模型类别。

你将在 `python/needle/nn/nn_transformer.py` 中实现一个 Transformer。

Transformer 由三个主要组件构成，你将分别实现它们：
1. 一个带掩码的多头注意力机制（masked multi-head attention mechanism），能够自适应地关注序列的不同时间步。
2. 一个残差块，由注意力层后接一个独立应用于每个时间步的两层神经网络组成。
3. 一个由数个堆叠的残差块组成的 Transformer 模型（在本作业中你将实现一个仅有解码器的 Transformer）。

![模型示意图](https://miro.medium.com/v2/1*ZCFSvkKtppgew3cc7BIaug.png)

上图是 Vaswani 等 2017 年论文中的 Transformer 架构图。你将实现的 Transformer 版本与其几乎相同，但在每个残差块的开头应用了层归一化（Layer Normalization），这种变体称为 [prenorm 变体](https://arxiv.org/abs/2002.04745)。

## 第一部分：实现多头注意力激活层

在这一子问题中，你将在 `python/needle/nn/nn_transformer.py` 中实现“基础”注意力激活层 `MultiHeadAttention` 的 `forward` 函数。该激活层接收三个输入：

多头查询 $Q \in R^{\mathcal{B \times H \times T \times D}}$，键 $K \in R^{\mathcal{B \times H \times T \times D}}$，值 $V \in R^{\mathcal{B \times H \times T \times D}}$

其中 $B$ 是批大小，$H$ 是注意力头数，$T$ 是序列长度，$D$ 是隐藏层维度。

注意力输出 $X \in R^{B \times H \times T \times D}$ 的计算方式如下：

$X = \text{softmax}\left(\frac{Q K^T}{\sqrt{D}}\right) V$

注意上述矩阵乘法是批处理的。这个功能在 needle 中尚未原生支持，因此我们在 `MultiHeadAttention` 中提供了一个便捷函数 `matmul` 用于批处理矩阵乘法。本部分的目标是根据输入的查询、键和值返回 $X$。

对于自回归 Transformer，该注意力应使用我们提供的函数 `self.create_causal_mask` 支持因果掩码（causal masking），以确保下一个 token 的预测仅依赖于之前的 token。具体而言，因果掩码是在 softmax 之前应用一个掩码，使得 softmax 概率在掩码后的矩阵 $\frac{Q K^T}{\sqrt{D}}$ 上计算。

此外，你的实现应对注意力 softmax $\text{softmax}\left(\frac{Q K^T}{\sqrt{D}}\right)$ 应用 dropout。你可以使用 `MultiHeadAttention` 模块的 `self.dropout` 函数。

重要的是，该层仅是一个激活函数，没有可训练变量（这些将在后面出现）。

完成实现后，用以下测试用例测试你的代码。

In [ ]:
!python3 -m pytest -l -v -k "attention_activation"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "hw4extra" -k "attention_activation"

## 第二部分：实现带有可训练参数的自注意力层

在这一子问题中，你将使用刚实现的 `MultiHeadAttention` 类，并将其封装到 `python/needle/nn/nn_transformer.py` 中 `Module` 的子类 `AttentionLayer` 中。

该层实现了 prenorm 下的自注意力（当 `self.forward` 调用中的 k 和 v 为 None 时）和交叉注意力（当 k 和 v 存在时）。我们已提供了包含相应层属性的骨架代码。你的任务是编写 `AttentionLayer` 的前向传播。注意你要实现的是多头注意力，注意力头数由 `AttentionLayer` 类的 `self.num_head` 属性给出。

给定输入 $Q \in R^{\mathcal{B \times T \times D'}}$，键 $K \in R^{\mathcal{B \times T \times D'}}$，值 $V \in R^{\mathcal{B \times T \times D'}}$，其中 $B$ 是批大小，$T$ 是序列长度，$D'$ 是嵌入维度。该层按以下顺序执行计算：

(1) 将查询、键和值映射到多头空间。

$Q' = \text{LayerNorm}_q (Q) \; W_q$

$K' = \text{LayerNorm}_k (K) \; W_k$

$V' = \text{LayerNorm}_v (V) \; W_v$

其中 $\text{LayerNorm}_q$、$\text{LayerNorm}_k$、$\text{LayerNorm}_v$ 分别是 prenorm `self.prenorm_q`、`self.prenorm_k` 和 `self.prenorm_v`。

(2) 从通道轴分解出头。

<p style="text-align: center;">$Q' \in R^{B \times T \times (HD)} \to Q' \in R^{B \times H \times T \times D}$

<p style="text-align: center;">$K' \in R^{B \times T \times (HD)} \to K' \in R^{B \times H \times T \times D}$

<p style="text-align: center;">$V' \in R^{B \times T \times (HD)} \to V' \in R^{B \times H \times T \times D}$

其中 $H$ 和 $D$ 分别是 `self.num_head` 和 `self.head_dim`。

(3) 计算多头注意力激活。

<p style="text-align: center;">$X = \text{softmax}\left(\frac{Q' (K')^T}{\sqrt{D}}\right) V'$

<p style="text-align: center;">$X \in R^{B \times H \times T \times D} \to X \in R^{B \times T \times H \times D}$

<p style="text-align: center;">$X \in R^{B \times T \times H \times D} \to X \in R^{B \times T \times (HD)}$

最后两步进行转置然后重塑，使隐藏状态具有正确的形状。

(4) 使用 `self.out_projection` 投影回层的输入空间

<p style="text-align: center;">$X' = X \; W_o$

本部分的目标是在 `AttentionLayer` 的 `self.forward` 调用中返回 $X$。为便于调试，你可以捕获内部 `MultiHeadAttention` 模块返回的 `probs` 变量，并将其存储到注意力层的一个属性中，例如 `self.probs`。

完成后，用以下测试用例测试你的层。

In [ ]:
!python3 -m pytest -l -v -k "attention_layer"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "hw4extra" -k "attention_layer"

## 第三部分：实现一个 prenorm 残差 Transformer 层

至此，你已经拥有了构建完整 Transformer 所需的全部部件。在这一子问题中，你将把注意力层与前馈网络组装成一个可堆叠的残差块。我们已在 `TransformerLayer` 类中提供了起始代码。

你需要在模块 `TransformerLayer` 的 `self.__init__` 调用中定义必要的类属性，并在 `self.forward` 中实现前向传播。你的 Transformer 层应支持对从上一阶段得到的 $X'$ 应用 dropout，然后再添加残差连接。按以下伪代码实现该层，并正确处理中间张量的形状：

x - 当前的隐藏状态序列

$x = x + \text{Dropout}(\text{Attention}(x))$
$x = x + \text{Dropout}(\text{Linear}_{2}(\text{Dropout}(\text{ReLU}(\text{Linear}_{1}(\text{LayerNorm1d}(x))))))$

对于 MLP，有两个线性层 $\text{Linear}_{1}$ 和 $\text{Linear}_{2}$：
- $\text{Linear}_{1}$：输入形状 `q_features`，输出形状 `hidden_size`
- $\text{Linear}_{2}$：输入形状 `hidden_size`，输出形状 `q_features`

完成后，运行以下测试用例。

In [ ]:
!python3 -m pytest -l -v -k "transformer_layer"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "hw4extra" -k "transformer_layer"

## 第四部分：实现 Transformer 模型

在本小节中，你将组合上一部分实现的残差 Transformer 层，构建完整的 Transformer 模型。在 `Transformer` 类中填写代码，通过定义一组包含从父类 `Transformer` 传递进来的适当参数的 `num_layers` 个 `TransformerLayer` 模块。然后实现 `Transformer` 的 `self.forward` 调用。

目前，你的 Transformer 层是置换不变的（permutation-invariant），无法区分每个 token 在序列中的位置。为打破这种对称性，你需要在 Transformer 中添加位置嵌入。

最初的 Transformer 论文使用正弦位置编码，并将其加到第一个 `TransformerLayer` 之前的输入嵌入上。这种方法效果很好，但现代 Transformer 中更常见的策略是学习位置嵌入。

为此，你应该使用 `needle.nn.Embedding`。在你的 Transformer 实现中，使用 homework 4 中的 `needle.nn.Embedding` 创建一个可学习的位置编码，其中 `num_embeddings` 设置为 `sequence_len`。给定一个输入序列，你应该创建一个张量，其中每个 token 对应其时间步 id（时间步递增，表示 token 在时间上的位置），然后像使用单词 id 一样使用它。

最后，将创建的位置编码加到输入 token 嵌入上，再送入你的 Transformer 层。

完成后，提交以下测试用例。

In [ ]:
!python3 -m pytest -l -v -k "transformer_model"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "hw4extra" -k "transformer_model"

现在，你可以在 Penn Treebank 数据集上训练一个 Transformer 语言模型：

注意：确保在 `apps/models.py` 的 `LanguageModel` 类中初始化一个 Transformer 模型；另外，对于 Transformer，最后的线性头 `self.linear` 的输入维度应为 `embedding_size`。

In [ ]:
import needle as ndl
sys.path.append('./apps')
from models import LanguageModel
from simple_ml import train_ptb, evaluate_ptb

device = ndl.cuda()
corpus = ndl.data.Corpus("data/ptb")
train_data = ndl.data.batchify(corpus.train, batch_size=256, device=device, dtype="float32")
model = LanguageModel(20, len(corpus.dictionary), hidden_size=32, num_layers=1, seq_model='transformer', seq_len=20, device=device)
train_ptb(model, train_data, seq_len=20, n_epochs=10, device=device, lr=0.003, optimizer=ndl.optim.Adam)
evaluate_ptb(model, train_data, seq_len=20, device=device)